# Introduction: Quantum Computing and Error Mitigation

Welcome to the **CU Boulder Qiskit Fall Fest Quantum Error Management Workshop!**

Quantum computers use the principles of quantum mechanics to perform computations, but real quantum hardware is imperfect. Noise from interactions with the environment, imperfect quantum gates, and measurement errors can all affect the results of quantum computations. Understanding how to manage these errors is an important step toward building useful quantum computers.

This notebook is designed to introduce you to the fundamentals of quantum circuits and provide hands-on experience with **quantum error suppression and mitigation** using [Qiskit](https://www.ibm.com/quantum/qiskit), an open-source quantum computing software framework.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand quantum gates and circuits:** Learn how common quantum gates manipulate quantum states and how to combine them into quantum circuits.
2. **Build and simulate quantum circuits:** Use Qiskit to construct circuits, simulate their execution, and interpret measurement results.
3. **Understand quantum noise:** Explore how errors affect quantum computations and why real hardware produces different results from ideal simulations.
4. **Explore error management techniques:** Implement two introductory approaches to managing quantum errors:
   - **Circuit-based error mitigation (ZNE):** Insert additional gates to systematically increase noise, then use classical extrapolation to estimate the ideal expectation value.
   - **Statistical error mitigation:** Use classical postprocessing to improve estimates obtained from noisy quantum measurements.

## How to Use This Notebook

**This notebook is not graded!** It is intended purely as a learning resource to help you build the skills and intuition needed for the two quantum error mitigation challenge notebooks provided as part of Fall Fest.

Throughout the notebook, you will encounter explanations, code examples, and short exercises. We encourage you to experiment with the code, make predictions before running simulations, and explore what happens when you change different parts of a circuit.

We assume you have completed the introductory linear algebra bootcamp, but **no prior experience with quantum computing or Qiskit is required**.

**The goal is not to finish as quickly as possible, but to develop enough understanding and confidence to tackle the two follow-up error mitigation challenges.**

## 0. Installations.

We recommend using Google Colab via your Colorado Boulder Gmail accounts for this notebook- the provided setup cell should enable use of all required packages.

In [1]:
%pip install -q \
    "qiskit[visualization]==2.5.2" \
    "qiskit-aer==0.18.2" \
    numpy matplotlib pylatexenc
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

print("Qiskit setup successful!")

Qiskit setup successful!



## 1. Quantum Gates and Circuit Construction

Quantum computations are performed by applying **quantum gates** to qubits. Just as classical logic gates manipulate bits, quantum gates transform quantum states. Mathematically, these gates are represented by unitary matrices acting on statevectors.

In Qiskit, we construct quantum circuits using the `QuantumCircuit` class. By default, each qubit begins in the state $|0\rangle$. We can then apply gates to manipulate its state.

Some common quantum gates include:

| Gate | Description | Qiskit Method |
|---|---|---|
| Pauli-X ($X$) | Flips $\lvert 0\rangle \leftrightarrow \lvert 1\rangle$ | `qc.x(0)` |
| Hadamard ($H$) | Creates superpositions of basis states | `qc.h(0)` |
| Pauli-Z ($Z$) | Changes the sign of the $\lvert 1\rangle$ amplitude | `qc.z(0)` |
| CNOT ($CX$) | Flips a target qubit when the control is $\lvert 1\rangle$ | `qc.cx(0, 1)` |


**Useful Qiskit API References:**
- [QuantumCircuit Documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.QuantumCircuit)
- [Pauli-X Gate](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.XGate)
- [Hadamard Gate](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.HGate)
- [Pauli-Z Gate](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.ZGate)
- [CNOT Gate](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.CXGate)

Let's build our first quantum circuit!


In [ ]:
# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Display tools
from IPython.display import display

In [ ]:

# Exercise 1.1: Your First Quantum Circuit
#
# Create a single-qubit circuit and apply the following gates:
#
# 1. Apply an X gate to qubit 0.
# 2. Apply a Hadamard (H) gate to qubit 0.
#
# Before running the cell, predict the final statevector!
# Recall that all qubits start in the |0> state.

# Create a quantum circuit with one qubit
qc = QuantumCircuit(1)

# TODO: Apply an X gate to qubit 0


# TODO: Apply an H gate to qubit 0


# Display your circuit
print("Your quantum circuit:")
display(qc.draw("text"))

# Calculate and display the final statevector
final_state = Statevector.from_instruction(qc)

print("Final statevector:")
display(final_state.draw("latex"))


## 2. Multi-Qubit Circuits and Quantum Measurement

Now that we've introduced basic quantum gates, let's combine them to construct more interesting quantum states!

One of the most important properties of quantum computing is **entanglement**. When qubits are entangled, their quantum states cannot be described independently of one another. Measuring one qubit gives us information about the possible measurement outcomes of the others.

In this section, we'll construct several entangled quantum states and explore how to measure them using Qiskit.

### Exercise 2.1: Constructing a Bell State

Our first goal is to construct the following two-qubit **Bell state**:

$$
|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)
$$

We can create this state using two gates:

1. **Hadamard (H):** Apply H to qubit 0, placing it in an equal superposition of $|0\rangle$ and $|1\rangle$.
2. **CNOT (CX):** Use qubit 0 as the control and qubit 1 as the target, entangling the two qubits.

### Measuring Quantum States

When we measure a quantum state in the computational basis, we obtain a classical bitstring. The probability of measuring each bitstring is the squared magnitude of its amplitude.

For our Bell state, we expect:

- **50% probability** of measuring `00`.
- **50% probability** of measuring `11`.

We can estimate these probabilities by running the circuit many times. Each execution is called a **shot**.

Qiskit provides the `measure_all()` method to measure every qubit in a circuit. We can then run the circuit using the `AerSimulator` and visualize the resulting measurement counts.

**Before running the code below:** What do you think will happen if we run the circuit 1,024 times? Will we measure `00` exactly 512 times?


In [ ]:

# Additional imports for measurement and simulation
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# Step 1: Create a two-qubit circuit
bell = QuantumCircuit(2)

# Step 2: Apply a Hadamard gate to qubit 0
bell.h(0)

# Step 3: Apply a CNOT with control=0 and target=1
bell.cx(0, 1)

# Display the circuit and its statevector
display(bell.draw("text"))
display(Statevector.from_instruction(bell).draw("latex"))

# Step 4: Add measurements to a copy of the circuit
bell_measured = bell.copy()
bell_measured.measure_all()

# Step 5: Run the circuit for 1024 shots
simulator = AerSimulator()
result = simulator.run(bell_measured, shots=1024).result()
counts = result.get_counts()

# Step 6: Display the measurement results
print("Measurement counts:", counts)
display(plot_histogram(counts))



### Exercise 2.2: Bell States and Relative Phases

There are actually four different Bell states! In the previous exercise, we constructed:

$$
|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)
$$

Now, your goal is to construct two more Bell states:

$$
|\Psi^+\rangle = \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle)
$$

$$
|\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)
$$

Notice that these two states differ only by a **relative phase** (the minus sign).

**Your task:**

1. Starting from the Bell state in Exercise 2.1, use a Pauli-X gate to construct $|\Psi^+\rangle$.
2. Apply an additional Pauli-Z gate to transform $|\Psi^+\rangle$ into $|\Psi^-\rangle$.
3. Display the statevectors of both circuits to verify your results.
4. Measure both qubits and simulate 1,024 shots for each circuit.
5. Compare the resulting measurement histograms.

**Hint:** Remember that X flips a qubit, while Z changes the sign of its $|1\rangle$ amplitude.

**Questions to consider:**
- Which bitstrings should appear in the measurement results?
- Do the two states produce different measurement histograms?
- If the statevectors are different, why might their measurement statistics be identical?
- Does this mean the relative phase has no physical significance?



In [ ]:

# Exercise 2.2: Construct |Psi+> and |Psi->

# Part A: Construct |Psi+>
# Start with the Bell state from Exercise 2.1
psi_plus = bell.copy()

# TODO: Apply an X gate to obtain (|01> + |10>)/sqrt(2)


# Part B: Construct |Psi->
# Start with the circuit you just created
psi_minus = psi_plus.copy()

# TODO: Apply a Z gate to obtain (|01> - |10>)/sqrt(2)


# Display both circuits and their statevectors
for name, circuit in [("Psi+", psi_plus), ("Psi-", psi_minus)]:
    print(f"\n{name} circuit:")
    display(circuit.draw("text"))

    print(f"{name} statevector:")
    display(Statevector.from_instruction(circuit).draw("latex"))

# Part C: Compare measurement results
for name, circuit in [("Psi+", psi_plus), ("Psi-", psi_minus)]:

    measured = circuit.copy()

    # TODO: Measure all qubits


    # TODO: Simulate 1024 shots and retrieve the counts


    # TODO: Plot a histogram of the measurement results



### Exercise 2.3: Scaling Up — The GHZ State

So far, we've worked with two-qubit entangled states. But entanglement is not limited to two qubits!

A **Greenberger-Horne-Zeilinger (GHZ) state** is an entangled state that can involve three or more qubits.

For four qubits, the GHZ state is:

$$
|\mathrm{GHZ}_4\rangle = \frac{1}{\sqrt{2}}(|0000\rangle + |1111\rangle)
$$

Notice that this looks very similar to our first Bell state, but now we have four qubits instead of two.

**Your task:**

1. Create a quantum circuit with four qubits.
2. Use Hadamard and CNOT gates to prepare the GHZ state.
3. Display the final statevector and verify that it matches the target state.
4. Measure all four qubits and simulate 1,024 shots.
5. Plot your measurement results.

**Hint:** Start by applying a Hadamard gate to qubit 0. Then think about how you could use a sequence of CNOT gates to entangle the remaining qubits.

**Questions to consider:**
- How many CNOT gates do you need?
- Which bitstrings do you expect to measure?
- How does the number of possible measurement outcomes compare to the number of outcomes with nonzero probability?


In [ ]:

# Exercise 2.3: Construct a four-qubit GHZ state

# Step 1: Create a circuit with four qubits
ghz = QuantumCircuit(4)

# TODO: Apply a Hadamard gate to qubit 0


# TODO: Apply CNOT gates to entangle all four qubits



# Step 2: Display the circuit and final statevector
display(ghz.draw("text"))
display(Statevector.from_instruction(ghz).draw("latex"))

# Step 3: Add measurements
ghz_measured = ghz.copy()

# TODO: Measure all four qubits


# Step 4: Run 1024 shots
# TODO: Execute the circuit and retrieve measurement counts



# Step 5: Plot your measurement results
# TODO: Display the histogram




## 3. Quantum Observables and Noise

So far, we've prepared quantum states and measured their computational-basis probabilities. However, many quantum algorithms are not interested in the probability of measuring a particular bitstring. Instead, they aim to calculate the **expectation value of a quantum observable**.

An observable is a Hermitian operator representing a measurable physical quantity. Given a quantum state $|\psi\rangle$ and an observable $O$, its expectation value is

$$
\langle O\rangle = \langle\psi|O|\psi\rangle.
$$

Expectation values are especially important in applications like quantum chemistry, quantum simulation, and optimization, where we might want to estimate a system's energy, magnetization, or correlations between qubits.

In this section, we'll focus on **Pauli observables**, constructed from the matrices $I$, $X$, $Y$, and $Z$.

### Measuring Pauli Observables

Quantum computers typically perform measurements in the computational (Z) basis. A measurement of Z gives:

- Outcome `0` corresponds to eigenvalue $+1$.
- Outcome `1` corresponds to eigenvalue $-1$.

Therefore,

$$
\langle Z\rangle = P(0)-P(1).
$$

We can also measure X and Y by applying **basis rotations** before measuring:

- **Z measurement:** Measure directly.
- **X measurement:** Apply H, then measure.
- **Y measurement:** Apply S-dagger (Sdg), then H, then measure.

For multi-qubit observables, we multiply the measurement eigenvalues together.

**Important:** Qiskit writes two-qubit Pauli operators in the order `q1 q0`. For example, `ZI` means Z acts on qubit 1 and I acts on qubit 0.

### What About Noise?

Real quantum computers experience errors during gate operations, interactions with their environment, and measurement. These errors can cause experimentally estimated expectation values to differ from their ideal values.

We'll use Qiskit's `AerSimulator` to compare:

1. **Ideal simulation:** The circuit executes without physical noise.
2. **Noisy simulation:** The circuit experiences simulated gate errors and measurement errors.

Our noise model is intentionally exaggerated so the effects are easy to observe.

**Useful references:**
- [Measuring Pauli Observables](https://quantum.cloud.ibm.com/docs/en/guides/specify-observables-pauli)
- [Qiskit Aer Noise Models](https://quantum.cloud.ibm.com/docs/en/guides/build-noise-models)


In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.visualization import plot_histogram

# Construct a simple noise model
noise_model = NoiseModel()

# 8% depolarizing noise on single-qubit gates
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.08, 1),
    ["h", "x", "s", "sdg"]
)

# 12% depolarizing noise on CNOT gates
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.12, 2),
    ["cx"]
)

# Each measurement has a 6% probability of flipping
readout_error = ReadoutError([
    [0.94, 0.06],
    [0.06, 0.94]
])
noise_model.add_all_qubit_readout_error(readout_error)

# Create the simulators
ideal_sim = AerSimulator()
noisy_sim = AerSimulator(noise_model=noise_model)

SHOTS = 2048

# Helper: estimate a Pauli expectation value from counts.
# The circuit must already include the correct basis rotations.
def expectation_from_counts(counts, pauli):
    total = sum(counts.values())
    expectation = 0.0

    for bitstring, count in counts.items():
        eigenvalue = 1

        for bit, operator in zip(bitstring, pauli):
            if operator != "I" and bit == "1":
                eigenvalue *= -1

        expectation += eigenvalue * count / total

    return expectation


# Helper: compare ideal and noisy measurement results
def compare_results(measured_circuit, pauli):
    ideal_counts = ideal_sim.run(
        measured_circuit, shots=SHOTS
    ).result().get_counts()

    noisy_counts = noisy_sim.run(
        measured_circuit, shots=SHOTS
    ).result().get_counts()

    print(f"Ideal <{pauli}>:",
          round(expectation_from_counts(ideal_counts, pauli), 4))
    print(f"Noisy <{pauli}>:",
          round(expectation_from_counts(noisy_counts, pauli), 4))

    display(plot_histogram(
        [ideal_counts, noisy_counts],
        legend=["Ideal", "Noisy"]
    ))



### Example 3.1: Measuring the ZI Observable

Let's start by measuring the expectation value of the observable $ZI$, which corresponds to measuring Z on qubit 1 and applying the identity operator to qubit 0.

We'll prepare the following two-qubit state:

$$
|\psi\rangle = |1\rangle \otimes |+\rangle
= \frac{|10\rangle + |11\rangle}{\sqrt{2}}.
$$

Recall that the Pauli-Z operator has eigenvalues $+1$ for $|0\rangle$ and $-1$ for $|1\rangle$. Since qubit 1 is always in the state $|1\rangle$, we expect:

$$
\boxed{\langle ZI\rangle = -1}
$$

To calculate this expectation value experimentally, we measure the circuit multiple times and use the resulting measurement counts to estimate:

$$
\langle ZI\rangle = P(q_1=0) - P(q_1=1).
$$

**In the code below, we will:**

1. Prepare our two-qubit quantum state.
2. Measure both qubits in the computational basis.
3. Execute the circuit using both ideal and noisy simulators.
4. Calculate and compare the resulting expectation values.

**Pay attention to:** How does the noisy expectation value differ from the ideal value? What happens to the measurement histogram when noise is introduced?


In [ ]:
# Example 3.1: Measuring <ZI>

# Prepare |1+> = (|10> + |11>)/sqrt(2)
qc_zi = QuantumCircuit(2)
qc_zi.x(1)
qc_zi.h(0)

print("State preparation:")
display(qc_zi.draw("text"))

print("Statevector:")
display(Statevector.from_instruction(qc_zi).draw("latex"))

# We want to measure Z on qubit 1.
# Z is already diagonal in the computational basis,
# so no additional gates are required.

measured_zi = qc_zi.copy()
measured_zi.measure_all()

print("Measurement circuit:")
display(measured_zi.draw("text"))

# Expected ideal value: <ZI> = -1
compare_results(measured_zi, "ZI")



### Exercise 3.2: Measuring a Two-Qubit Correlation — ZX

Now let's measure a more interesting observable:

$$
\langle ZX\rangle = \langle\psi|Z\otimes X|\psi\rangle.
$$

This observable measures a correlation between the Z component of qubit 1 and the X component of qubit 0.

**Your task:**

1. Start from the Bell state constructed in Exercise 2.1.
2. Apply an H gate to qubit 0 to prepare the desired state.
3. Apply the appropriate basis rotation to measure X on qubit 0.
4. Measure both qubits.
5. Compare ideal and noisy expectation values.

**Expected ideal result:**

$$
\boxed{\langle ZX\rangle = +1}
$$

In [ ]:

# Exercise 3.2: Measuring <ZX>

# Start from our original Bell state
qc_zx = bell.copy()

# TODO: Apply an H gate to qubit 0 to prepare
# a state with ideal <ZX> = +1


# Display your prepared state
display(qc_zx.draw("text"))
display(Statevector.from_instruction(qc_zx).draw("latex"))

# Construct the measurement circuit
measured_zx = qc_zx.copy()
measured_zx.barrier()

# TODO: Rotate qubit 0 into the X measurement basis


# TODO: Measure all qubits


# Display your measurement circuit
display(measured_zx.draw("text"))

# Expected ideal value: <ZX> = +1
compare_results(measured_zx, "ZX")


### Exercise 3.3: Measuring a Y Observable — IY

Finally, let's measure an observable involving the Pauli-Y matrix.

Our target state is

$$
|\psi\rangle = |0\rangle\otimes|+i\rangle
$$

where

$$
|+i\rangle = \frac{|0\rangle+i|1\rangle}{\sqrt{2}}.
$$

The phase gate S acts as

$$
S = \begin{pmatrix}1&0\\0&i\end{pmatrix}.
$$

Therefore, applying H followed by S to $|0\rangle$ prepares $|+i\rangle$.

**Your task:**

1. Prepare the target state using H and S gates.
2. Apply the appropriate basis rotations to measure Y on qubit 0.
3. Measure both qubits.
4. Compare ideal and noisy results.

**Expected ideal result:**

$$
\boxed{\langle IY\rangle = +1}
$$

**Questions to consider:**
- Why do we need additional gates to measure X and Y?
- Why are noisy expectation values different from ideal values?
- Do noise-induced errors always decrease an expectation value?
- What happens when you increase the number of shots? Does increasing the number of shots eliminate the effects of noise?


In [ ]:

# Exercise 3.3: Measuring <IY>

# Create a two-qubit circuit
qc_iy = QuantumCircuit(2)

# TODO: Prepare |0> tensor |+i>

# Display your state
display(qc_iy.draw("text"))
display(Statevector.from_instruction(qc_iy).draw("latex"))

# Construct the measurement circuit
measured_iy = qc_iy.copy()
measured_iy.barrier()

# TODO: Rotate qubit 0 into the Y measurement basis


# TODO: Measure all qubits


# Display the measurement circuit
display(measured_iy.draw("text"))

# Expected ideal value: <IY> = +1
compare_results(measured_iy, "IY")



## 4. Quantum Error Mitigation: Zero-Noise Extrapolation

In the previous section, we observed that noise can cause expectation values measured on quantum computers to differ from their ideal values.

One approach to addressing this problem is **Zero-Noise Extrapolation (ZNE)**. Rather than trying to eliminate noise directly, ZNE estimates what the result of a quantum computation would have been **if there were no noise**.

### How Does ZNE Work?

The basic idea is surprisingly simple:

1. **Run a noisy circuit** and measure an observable.
2. **Artificially increase the noise** in the circuit and repeat the measurement.
3. **Repeat at several noise levels** to observe how the expectation value changes.
4. **Extrapolate backward** to estimate the expectation value at zero noise.

But how can we increase noise without changing the computation?

### Gate Folding

One approach is called **gate folding**.

Recall that the CNOT gate is its own inverse:

$$
CX^2 = I.
$$

Therefore, applying three CNOT gates in a row is mathematically equivalent to applying one:

$$
CX^3 = CX.
$$

Similarly,

$$
CX^5 = CX.
$$

On an ideal quantum computer, all three implementations produce the same result. However, on a noisy quantum computer, executing additional gates introduces additional opportunities for errors.

This allows us to construct circuits with approximately 1x, 3x, and 5x the CNOT-gate noise exposure, while preserving the ideal computation.

We can then fit a curve to the measured expectation values as a function of the noise scale $\lambda$:

$$
\langle O\rangle(\lambda) \approx a+b\lambda+c\lambda^2.
$$

Evaluating this curve at $\lambda=0$ gives an estimate of the zero-noise expectation value:

$$
\langle O\rangle_{\mathrm{ZNE}} \approx a.
$$

### Exercise 4.1: ZNE on a GHZ State

Let's return to the four-qubit GHZ state from Exercise 2.3:

$$
|\mathrm{GHZ}_4\rangle =
\frac{|0000\rangle+|1111\rangle}{\sqrt{2}}.
$$

We'll measure the observable

$$
O = X\otimes X\otimes X\otimes X.
$$

For an ideal GHZ state,

$$
\boxed{\langle XXXX\rangle = +1}.
$$

Remember that measuring X requires applying a Hadamard gate before computational-basis measurement.

**Your task:**

1. Construct a four-qubit GHZ circuit using H and CNOT gates.
2. Implement gate folding by repeating each CNOT gate 1, 3, or 5 times.
3. Measure $\langle XXXX\rangle$ for each folded circuit using a noisy simulator.
4. Plot the measured expectation values against the noise scale.
5. Use `numpy.polyfit()` to fit a quadratic curve and extrapolate to zero noise.
6. Compare your ZNE estimate with the original noisy result and the ideal value of $+1$.

**Questions to consider:**
- Why do we only use odd numbers of CNOT gates?
- Why does the measured expectation value change even though the ideal circuit is unchanged?
- Is the extrapolated estimate closer to the ideal result?
- What are the limitations of extrapolating from noisy data?

**Note:** For this exercise, our noise model includes only CNOT-gate errors, so gate folding gives us a controlled way to increase noise exposure. Real quantum hardware also experiences other types of noise, which ZNE may not fully remove.


In [ ]:
# Exercise 4.1: Zero-Noise Extrapolation on a GHZ State

import numpy as np
import matplotlib.pyplot as plt

# Set up a noisy simulator (CNOT errors only)
zne_noise = NoiseModel()
zne_noise.add_all_qubit_quantum_error(
    depolarizing_error(0.06, 2), ["cx"]
)

zne_sim = AerSimulator(noise_model=zne_noise)
ZNE_SHOTS = 20000


# Part A: Construct a folded GHZ circuit

def folded_ghz(scale):
    qc = QuantumCircuit(4)

    # Start GHZ preparation
    qc.h(0)

    # TODO: Build the GHZ state using CNOT gates.
    # Repeat each CNOT 'scale' times.
    # Hint: try the pairs (0,1), (1,2), (2,3).



    # TODO: Apply basis rotations to measure XXXX



    # TODO: Measure all qubits


    return qc


# Part B: Measure at different noise scales

scales = [1, 3, 5]
expectations = []

for scale in scales:
    qc = folded_ghz(scale)

    counts = zne_sim.run(
        qc, shots=ZNE_SHOTS
    ).result().get_counts()

    # TODO: Calculate <XXXX> using
    # expectation_from_counts() from Section 3

    value = None

    expectations.append(value)


# Part C: Extrapolate to zero noise

# TODO: Fit a quadratic polynomial using np.polyfit()


# TODO: Evaluate the polynomial at scale = 0
# Hint: np.polyval()


# Part D: Visualize your results

# TODO: Plot measured expectation values vs noise scale.
# Add the extrapolated value at scale = 0.
# Draw a horizontal line at the ideal value (+1).


# TODO: Compare the original noisy expectation
# with the ZNE estimate.



### Beyond Zero-Noise Extrapolation

Congratulations! You've implemented your first quantum error mitigation technique.

In this exercise, we used **Zero-Noise Extrapolation (ZNE)** to estimate the ideal expectation value of a noisy quantum circuit. By inserting additional gates, we deliberately increased the noise without changing the ideal computation. We then used classical extrapolation to estimate the result at zero noise.

Importantly, ZNE does not actually remove errors from the quantum computer. Instead, it attempts to **improve our estimate of the ideal result**. ZNE also requires additional circuit executions, and its accuracy depends on the quality of our noise-scaling model and the statistical uncertainty in our measurements.

### Other Quantum Error Management Techniques

ZNE is just one of many techniques researchers use to manage errors in quantum computations! Other approaches include:

- **[Dynamical Decoupling](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques):** Inserts carefully chosen sequences of gates during idle periods to suppress certain unwanted interactions with the environment.

- **[Pauli Twirling / Randomized Compiling](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques):** Introduces randomized gate sequences that preserve the ideal computation while transforming the structure of the noise. This can make certain errors easier to manage.

- **[Probabilistic Error Cancellation (PEC)](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques):** Uses information about the hardware's noise to combine results from many noisy circuit executions into an estimate of the ideal result. This can be powerful, but may require a large number of additional measurements.

These techniques have different advantages, limitations, and computational costs. Some attempt to **suppress errors during execution**, while others use additional experiments and classical calculations to **mitigate their effects**.

### What About Classical Postprocessing?

So far, we've focused on modifying quantum circuits or executing additional circuit variants. However, another important approach is to use **classical postprocessing of measurement results**.

For example, suppose a quantum computer occasionally reports `1` when the true measurement outcome was `0`, or vice versa.

If we can characterize how often these measurement errors occur, we may be able to correct for their effects statistically, without modifying the quantum circuit we're trying to evaluate.

This is called **Measurement Error Mitigation**, and it's the subject of our final exercise!

For additional reading, check out IBM's [Error Mitigation and Suppression Guide](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques).



## 5. Quantum Error Mitigation: Measurement Error Mitigation

In the previous exercise, we used Zero-Noise Extrapolation to improve our estimates of quantum observables by running modified versions of a noisy circuit.

For our final exercise, we'll explore a different approach: **Measurement Error Mitigation (MEM)**.

### What Are Measurement Errors?

When we measure a qubit, the quantum computer attempts to determine whether its state corresponds to $|0\rangle$ or $|1\rangle$. However, real quantum hardware sometimes reports the wrong outcome.

For example:
- A qubit in state $|0\rangle$ might incorrectly be measured as `1`.
- A qubit in state $|1\rangle$ might incorrectly be measured as `0`.

These errors distort the measured probability distribution, which in turn affects our estimates of quantum observables.

### How Does Measurement Error Mitigation Work?

The key idea is that **we can characterize measurement errors by preparing states whose measurement outcomes we already know**.

For a single qubit, we can construct a **calibration matrix**:

$$
M =
\begin{pmatrix}
P(0|0) & P(0|1) \\
P(1|0) & P(1|1)
\end{pmatrix}
$$

Here, $P(i|j)$ represents the probability of measuring outcome $i$ when we prepare state $|j\rangle$.

The calibration matrix relates the true and observed probability distributions:

$$
\vec{p}_{\mathrm{observed}} = M\vec{p}_{\mathrm{true}}.
$$

If we know $M$, we can estimate the original probability distribution using:

$$
\vec{p}_{\mathrm{mitigated}} = M^{-1}\vec{p}_{\mathrm{observed}}.
$$

This allows us to **correct measurement statistics using classical postprocessing**, without modifying the quantum circuit we're trying to evaluate.

### Exercise 5.1: Correcting Noisy Measurement Results

Let's return to the two-qubit state from Example 3.1:

$$
|\psi\rangle = \frac{|10\rangle+|11\rangle}{\sqrt{2}}.
$$

Recall that the ideal expectation value is:

$$
\boxed{\langle ZI\rangle = -1}.
$$

**Your task:**

1. **Calibrate the measurement errors:** Prepare the known states $|0\rangle$ and $|1\rangle$, measure them repeatedly, and use the results to estimate a $2\times2$ calibration matrix.
2. **Run a noisy circuit:** Prepare the state from Example 3.1 and measure both qubits using our noisy simulator.
3. **Apply classical postprocessing:** Use your calibration matrix to correct the observed probability distribution for qubit 1.
4. **Calculate the corrected expectation value:** Use the mitigated probabilities to estimate $\langle ZI\rangle$.
5. **Compare your results:** How close are the unmitigated and mitigated expectation values to the ideal value of $-1$?

**Hints:**
- Remember that $\langle Z\rangle=P(0)-P(1)$.
- You can use `numpy.linalg.inv()` or `numpy.linalg.solve()` to apply the calibration correction.
- You only need the measurement statistics for qubit 1 to calculate $\langle ZI\rangle$.

**Questions to consider:**
- Does measurement error mitigation improve your estimate?
- Why do we need additional circuit executions to construct the calibration matrix?
- Can measurement error mitigation correct errors that occurred during quantum gate operations?
- What might happen if our calibration matrix is inaccurate?

**Important:** Measurement error mitigation only addresses measurement errors. It does not correct errors introduced during state preparation or gate operations. Additionally, statistical uncertainty can make the corrected probabilities imperfect. In real applications, additional techniques may be needed to ensure the corrected distribution remains physically valid.


In [ ]:

# Exercise 5.1: Measurement Error Mitigation

import numpy as np

# We'll reuse the noise model and simulators from Section 3.
# For calibration, isolate readout noise so that errors in
# the X gate used to prepare |1> don't affect our calibration.

cal_noise = NoiseModel()
cal_noise.add_all_qubit_readout_error(readout_error)
cal_sim = AerSimulator(noise_model=cal_noise)

MEM_SHOTS = 10000


# Part A: Calibrate measurement errors
# Prepare and measure the known states |0> and |1>.

cal_0 = QuantumCircuit(1)
cal_1 = QuantumCircuit(1)

# TODO: Prepare |1> in cal_1


# TODO: Measure both calibration circuits


# TODO: Execute both circuits using cal_sim
# and retrieve the measurement counts

counts_0 = None
counts_1 = None


# Part B: Construct the calibration matrix
# Rows: measured outcomes 0, 1
# Columns: prepared states |0>, |1>

# TODO: Calculate P(0|0), P(1|0), P(0|1), P(1|1)

M = None

print("Calibration matrix:")
print(M)


# Part C: Measure our noisy circuit

# Reuse the state preparation from Example 3.1
# |psi> = (|10> + |11>)/sqrt(2)

measured_circuit = qc_zi.copy()
measured_circuit.measure_all()

counts = noisy_sim.run(
    measured_circuit, shots=MEM_SHOTS
).result().get_counts()

print("Noisy measurement counts:", counts)

# TODO: Calculate the observed probabilities
# P(q1=0) and P(q1=1).
# Hint: Qiskit displays q1 as the LEFTMOST bit
# in a two-qubit measurement string.

p_observed = None


# Part D: Apply measurement error mitigation

# TODO: Solve M @ p_mitigated = p_observed
# Hint: np.linalg.solve()

p_mitigated = None


# Part E: Compare expectation values

# TODO: Calculate <ZI> from the observed probabilities
expectation_noisy = None

# TODO: Calculate <ZI> from the mitigated probabilities
expectation_mitigated = None

print("Ideal <ZI>:", -1.0)
print("Noisy <ZI>:", expectation_noisy)
print("Mitigated <ZI>:", expectation_mitigated)


# Part F: Visualization (optional)

# TODO: Plot the ideal, noisy, and mitigated
# expectation values on a bar chart.




### Other Postprocessing-Based Error Mitigation Techniques

Measurement error mitigation is one example of using classical computation to improve the accuracy of quantum results. Researchers have developed several other approaches that use measurement data and classical postprocessing:

- **[Symmetry Verification and Postselection](https://doi.org/10.1103/PhysRevA.98.062339):** Many quantum states obey known physical constraints, such as conservation of particle number or parity. If a measurement violates one of these constraints, it may indicate that an error occurred. We can use these constraints to discard certain erroneous results or adjust our estimates.

- **[Clifford Data Regression (CDR)](https://quantum-journal.org/papers/q-2021-11-26-592/):** Uses quantum circuits whose ideal results can be efficiently calculated on a classical computer. By comparing their ideal and noisy results, we can train a classical regression model to correct the results of more complicated quantum circuits.

- **[Probabilistic Error Cancellation (PEC)](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques):** Combines measurements from multiple noisy circuit implementations using carefully chosen classical weights to estimate the ideal result. Unlike simple postprocessing of one circuit's outcomes, PEC also requires additional quantum circuit executions.

These techniques differ in what information they require, which errors they can mitigate, and how many additional measurements or circuit executions they need.
